In [ ]:
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(ggdist)
library(ggforce)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(purrr)
library(broom)
library(Polychrome)
library(cowplot)
library(reshape2)
library(doParallel)

In [ ]:
# Custom function for flat violin plot
geom_flat_violin <- function(mapping = NULL, data = NULL, stat = "ydensity",
                             position = "dodge", trim = TRUE, scale = "area",
                             show.legend = NA, inherit.aes = TRUE, ...) {
  ggplot2::layer(
    data = data,
    mapping = mapping,
    stat = stat,
    geom = ggplot2::GeomPolygon,
    position = position,
    show.legend = show.legend,
    inherit.aes = inherit.aes,
    params = list(trim = trim, scale = scale, ...)
  )
}

In [ ]:
metadata <- read.csv("DOME-16S-Nasal-Metadata.csv")
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

In [ ]:
GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv")

In [ ]:
relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  # prevent divide by zero
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")

# Step 3: Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))

# Step 4: Merge with metadata
merged_GENUS <- right_join(metadata, relative_abundance_genus, by = "Sample.ID")

In [ ]:
# ----------------------------
# 1. Prepare abundance matrix
# ----------------------------
abund_matrix <- relative_abundance_genus
rownames(abund_matrix) <- abund_matrix$Sample.ID
abund_matrix$Sample.ID <- NULL
abund_matrix <- abund_matrix[rowSums(abund_matrix) > 0, ]  # remove zero-total rows

# ----------------------------
# 2. Filter metadata to match abundance matrix
# ----------------------------
nasal_metadata <- metadata[metadata$Sample.ID %in% rownames(abund_matrix), ]

In [ ]:
bray_dist <- vegdist(abund_matrix, method = "bray")
bray_matrix <- as.matrix(bray_dist)

In [ ]:
write.csv(bray_matrix, "16S-Nasal-Bray-Curtis-Distance-As-Matrix.csv", row.names = TRUE)

In [ ]:
# Convert to long format
bc_long <- as.data.frame(as.table(bray_matrix))
colnames(bc_long) <- c("Sample1", "Sample2", "BrayCurtis")

# Join metadata for each sample
bc_long <- bc_long %>%
  left_join(nasal_metadata %>% select(Sample.ID, Cohort, Site), by = c("Sample1" = "Sample.ID")) %>%
  rename(Cohort1 = Cohort, Site1 = Site) %>%
  left_join(nasal_metadata %>% select(Sample.ID, Cohort, Site), by = c("Sample2" = "Sample.ID")) %>%
  rename(Cohort2 = Cohort, Site2 = Site)

In [ ]:
bc_long <- bc_long %>%
  filter(Cohort1 != Cohort2) %>%  # only cross-cohort pairs
  mutate(
    CohortComparison = case_when(
      Cohort1 == "Farmer" & Cohort2 == "Cow" ~ "Farmer",
      Cohort1 == "Cow" & Cohort2 == "Farmer" ~ "Farmer",
      Cohort1 == "Non-Farmer" & Cohort2 == "Cow" ~ "Non-Farmer",
      Cohort1 == "Cow" & Cohort2 == "Non-Farmer" ~ "Non-Farmer"
    ),
    SiteComparison = ifelse(Site1 == Site2, "Same Site", "Different Site")
  ) %>%
  filter(!is.na(CohortComparison))


In [ ]:
# Keep only human–cow pairs (avoid double-counting)
bc_long <- bc_long %>%
  filter(
    (Cohort1 %in% c("Farmer", "Non-Farmer") & Cohort2 == "Cow") |
    (Cohort2 %in% c("Farmer", "Non-Farmer") & Cohort1 == "Cow")
  )

# Identify which sample is human vs cow
bc_long <- bc_long %>%
  mutate(
    HumanSample = if_else(Cohort1 %in% c("Farmer", "Non-Farmer"), Sample1, Sample2),
    HumanCohort = if_else(Cohort1 %in% c("Farmer", "Non-Farmer"), Cohort1, Cohort2),
    CowSample    = if_else(Cohort1 == "Cow", Sample1, Sample2),
    CowSite      = if_else(Cohort1 == "Cow", Site1, Site2)
  )

# Recalculate SiteComparison using the human site vs cow site
bc_long <- bc_long %>%
  mutate(
    HumanSite = if_else(Cohort1 %in% c("Farmer", "Non-Farmer"), Site1, Site2),
    SiteComparison = if_else(HumanSite == CowSite, "Same Site", "Different Site")
  )

# Average Bray–Curtis per human per site condition
bc_summary <- bc_long %>%
  group_by(HumanSample, HumanCohort, SiteComparison) %>%
  summarise(MeanBrayCurtis = mean(BrayCurtis, na.rm = TRUE), .groups = "drop")

bc_summary <- bc_summary %>%
  mutate(
    HumanCohort = factor(HumanCohort, levels = c("Farmer", "Non-Farmer")),
    SiteComparison = factor(SiteComparison, levels = c("Same Site", "Different Site"))
  )

In [ ]:
write.csv(bc_summary, "16S-Nasal-Bray-Curtis-Distance-Human-To-Cow.csv")

In [ ]:
comparison_groups <- list(
    c('Farmer.Cow.Same Site', 'Non-Farmer.Cow.Same Site'),
    c('Farmer.Cow.Different Site', 'Non-Farmer.Cow.Different Site'),
    c('Farmer.Cow.Same Site', 'Non-Farmer.Cow.Different Site'),
    c('Farmer.Cow.Different Site', 'Non-Farmer.Cow.Same Site')    
)

In [ ]:
# Figure 1B
bray_curtis_plot <- ggplot(bc_summary, aes(x = SiteComparison, y = MeanBrayCurtis, fill = HumanCohort)) +
  geom_boxplot(
    width = 0.4,
    position = position_dodge(width = 0.6),
    outlier.shape = NA
  ) +
  stat_halfeye(
    aes(fill = HumanCohort),
    width = 0.4,
    justification = -0.6,
    point_colour = NA,
    position = position_dodge(width = 0.6)
  ) +
  geom_jitter(
    aes(color = HumanCohort),
    size = 0.25,
    alpha = 0.75,
    position = position_jitterdodge(dodge.width = 0.6, jitter.width = 0.35)
  ) +
  scale_fill_manual(values = genus_colors) +
  scale_color_manual(values = genus_colors) +
  coord_cartesian(ylim = c(0.5, 1.1)) +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    strip.background = element_rect(fill = "black"),
    strip.text = element_text(color = "white", face = "bold", size = 10),
    axis.text.x = element_text(size = 10),
    axis.title.x = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
    panel.background = element_rect(fill = "white"),
    legend.title = element_blank(),
    legend.position = c(0.01, 0.02),
    legend.justification = c("left", "bottom"),
    text = element_text(family = "Helvetica", size = 12)
  ) +
  ylab("Bray-Curtis Distance to Cow Samples")

In [ ]:
bc_summary <- bc_summary %>%
  mutate(Group = paste(HumanCohort, "Cow", SiteComparison))

wilcox_results <- compare_means(
  MeanBrayCurtis ~ Group,
  data = bc_summary,
  method = "wilcox.test",
  comparisons = comparison_groups,
  p.adjust.method = "BH"
)
wilcox_results <- wilcox_results %>%
  rename(
    Metric = .y.,
    Group1 = group1,
    Group2 = group2,
    P_value = p,
    Adjusted_P = p.adj,
    Significance = p.signif,
    P_formatted = p.format,
    Method = method
  ) %>%
  select(Group1, Group2, P_value, P_formatted, Adjusted_P, Significance, everything())

bc_comparison_sample_counts <- bc_summary %>%
  group_by(Group) %>%
  summarise(N = n())

In [ ]:
write.csv(wilcox_results, "Bray-Curtis-To-Cow-Wilcoxon-Testing.csv", row.names = FALSE)

In [ ]:
write.csv(bc_comparison_sample_counts, "Bray-Curtis-To-Cow-Wilcoxon-Testing-Sample-Sizes.csv", row.names = FALSE)